CONTENTS:


In [28]:
import logging
import os
import time

import numpy as np
import pandas as pd
import requests

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint

In [2]:
# Configure logger.
hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Print system signature.
_LOG.info("%s", henv.get_system_signature()[0])

# Configure the notebook style.
hprint.config_notebook()

INFO  > cmd='/venv/lib/python3.9/site-packages/ipykernel_launcher.py -f /home/.local/share/jupyter/runtime/kernel-e300149a-1202-4de8-adca-ae480003db60.json'
INFO  # Git
  branch_name='CmTask10499_Scrap_the_money_20/20_attendees'
  hash='766469cba'
  # Last commits:
    * 766469cba shaunak01 checkpoint                                                        (   2 hours ago) Wed Oct 30 00:28:38 2024  (HEAD -> CmTask10499_Scrap_the_money_20/20_attendees, origin/CmTask10499_Scrap_the_money_20/20_attendees)
    * 93312fb5a shaunak01 checkpoint                                                        (  20 hours ago) Tue Oct 29 06:19:18 2024           
    * 173609479 shaunak01 checkpoint                                                        (  21 hours ago) Tue Oct 29 06:00:14 2024           
# Machine info
  system=Linux
  node name=dfb319716156
  release=5.15.0-1070-aws
  version=#76~20.04.1-Ubuntu SMP Mon Sep 2 12:20:36 UTC 2024
  machine=x86_64
  processor=x86_64
  cpu count=8
  cpu freq=

In [4]:
username = os.getenv("username")
password = os.getenv("password")
x_authorization = os.getenv("x-authorization")

The way to get authorizations and headers is to inspect the required ulrs and find GET-APIs which fetch us the data .

In [5]:
login_payload = {
    "username": username,
    "password": password,
}
login_url = "https://auth.money2020.com/u/login?state=hKFo2SBvWGgxWVlIRUpxeHdPeVBlVHJ2cEF4NmJ5ZVJ2WTBpb6Fur3VuaXZlcnNhbC1sb2dpbqN0aWTZIDNBbXBwOFZqSmFSZ2ZVVkxRemtmSE9GRVY5MDdHNldYo2NpZNkgWGlQeGhkQzRlMzRSYXlHelhjV3ZBNFVyYWcxTEV2WXU"
api_headers = {
    "x-authorization": x_authorization,
    "Content-Type": "application/json",
    "Accept": "application/json",
}

In [6]:
data_collection = []
max_retries = 3

with requests.Session() as session:
    session.post(login_url, data=login_payload)
    base_url = (
        "https://api-prod.grip.events/1/container/7381/search/extension/82807"
    )
    for page in range(1, 1001):
        url = f"{base_url}?order=asc&page={page}&sort=name"
        # Track retry attempts for the current page
        retries = 0
        # Initial backoff time (in seconds) if rate-limited
        backoff_time = 1
        # Retry logic: continue trying if the request fails, up to max_retries
        while retries < max_retries:
            response = session.get(url, headers=api_headers)
            print(f"Requesting page {page}: {response.status_code}")
            # Check for a rate limit error (HTTP 429)
            if response.status_code == 429:
                print("Rate limit exceeded. Waiting before retrying...")
                time.sleep(backoff_time)
                # The min() function caps the wait time to a maximum of 60 seconds
                # Double the wait time for rate limiting, capping it at 60 seconds to avoid excessive delays.
                backoff_time = min(backoff_time * 2, 60)
                # Increment retry counter and retry the request
                retries += 1
                continue
            try:
                result = response.json()
                # Stop if the request is unsuccessful (e.g., invalid response)
                if not result.get("success"):
                    print(f"Stopping at page {page} as request was unsuccessful.")
                    break

                data = result.get("data", [])
                if data:
                    data_collection.extend(data)
                    # Reset backoff time for next page
                    backoff_time = 1
                    break
                else:
                    # Retry if no data was returned for this page
                    print(f"No data found on page {page}, retrying...")
                    retries += 1
                    time.sleep(1)
            except requests.JSONDecodeError:
                print(f"Stopping at page {page} due to invalid JSON response.")
                print("Response content:", response.text)
                break
        # Wait 4 seconds before moving to the next page to avoid rate limits
        time.sleep(4)

Requesting page 2841: 200
Requesting page 2842: 200
Requesting page 2843: 200
Requesting page 2844: 200
Requesting page 2845: 200
Requesting page 2846: 200
Requesting page 2847: 200
Requesting page 2848: 200
Requesting page 2849: 200
Requesting page 2850: 200
Requesting page 2851: 200
Requesting page 2852: 200
Requesting page 2853: 200
Requesting page 2854: 200
Requesting page 2855: 200
Requesting page 2856: 200
Requesting page 2857: 200
Requesting page 2858: 200
Requesting page 2859: 200
Requesting page 2860: 200
Requesting page 2861: 200
Requesting page 2862: 200
Requesting page 2863: 200
Requesting page 2864: 200
Requesting page 2865: 200
Requesting page 2866: 200
Requesting page 2867: 200
Requesting page 2868: 200
Requesting page 2869: 200
Requesting page 2870: 200
Requesting page 2871: 200
Requesting page 2872: 200
Requesting page 2873: 200
Requesting page 2874: 200
Requesting page 2875: 200
Requesting page 2876: 200
Requesting page 2877: 200
Requesting page 2878: 200
Requesting p

KeyboardInterrupt: 

In [9]:
processed_data = []
for i in data_collection:
    location = i.get("location", "")
    processed_data.append(
        {
            "id": i.get("id", ""),
            "firstName": (
                i.get("first_name", "").strip() if i.get("first_name") else ""
            ),
            "lastName": (
                i.get("last_name", "").strip() if i.get("last_name") else ""
            ),
            "location": (
                None
                if not isinstance(location, str) or "error" in location
                else location.strip()
            ),
            "companyName": (
                i.get("company_name", "").strip() if i.get("company_name") else ""
            ),
            "jobTitle": (
                i.get("job_title", "").strip() if i.get("job_title") else ""
            ),
        }
    )

df = pd.DataFrame(processed_data)

In [10]:
df.tail(6)

,id,firstName,lastName,location,companyName,jobTitle
1354,11647128,Shahir,Daya,"Vancouver, Canada",Zafin,Chief Technology Officer
1355,11665158,Shahzad,Khan,"Laguna Beach, United States",Keyno Inc,Chief Strategy Officer
1356,12044033,Shai,Kleiman,"New York, United States",Citibank N.A.,"VP, Relationship Manger"
1357,11884840,Shai,Schiller,"Amsterdam, Netherlands",Hub Security Inc,Head of Corporate Strategy
1358,12095788,Shailendra,Bade,"Phoenix, United States",American Express,Director Engineering
1359,11599128,Shaine,Hirsh,"San Mateo, United States",forml,CTO


In [11]:
df.to_csv("money20-20_Attandees.csv_Requests", index=False)

In [17]:
df_new = pd.read_csv("money2020_data.csv")

In [18]:
df_new.head()

,id,firstName,lastName,location,companyName,jobTitle
0,12178574,Alan,Murray,NaN,Dow Jones & Co.,"President, Dow Jones Leadership Inst."
1,11936408,Aakriti,Beri,"Palo Alto, United States",Global Affairs Canada,Trade Commissioner
2,12087185,Aaron,Adams,"Lansing, United States",CU Solutions Group,EVP/COO
3,11871313,Aaron,Adolphson,"Santa Cruz , United States",Yodlee,"Principal Director, D&A Marketing"
4,11600052,Aaron,Axworthy,"Vancouver, Canada",Zafin,VP Strategic Sales


In [ ]:
contact_sharings = []
is_connections = []
emails = []
phone_numbers = []

with requests.Session() as session:
    session.post(login_url, data=login_payload)
    for i in df_new["id"]:
        print(i)
        headers = {
            "Accept": "*/*",
            "Accept-Encoding": "gzip, deflate, br, zstd",
            "Accept-Language": "en-US,en;q=0.5",
            "cache-control": "no-cache",
            "pragma": "no-cache",
            "x-authorization": "7d5af131-d25f-42e9-8289-d7f35f9deb34",
        }
        contact_url = f"https://api-prod.grip.events/1/container/7381/thing/{i}/contact_details"
        response = requests.get(contact_url, headers=headers)
        if response.status_code == 200:
            contact_response = response.json()
            # Extract fields, using None as default if missing
            contact_sharings.append(
                contact_response["data"].get("contact_sharing", None)
            )
            is_connections.append(
                contact_response["data"].get("is_connection", None)
            )
            emails.append(contact_response["data"].get("email", None))
            phone_numbers.append(
                contact_response["data"].get("phone_number", None)
            )
        elif response.status_code == 204:
            print("No content available for this request.")
            contact_sharings.append(None)
            is_connections.append(None)
            emails.append(None)
            phone_numbers.append(None)
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")
            contact_sharings.append(None)
            is_connections.append(None)
            emails.append(None)
            phone_numbers.append(None)

df_new["contact_sharing"] = contact_sharings
df_new["is_connection"] = is_connections
df_new["email"] = emails
df_new["phone_number"] = phone_numbers

In [23]:
df_new.head(20)

,id,firstName,lastName,location,companyName,jobTitle,contact_sharing,is_connection,email,phone_number
0,12178574,Alan,Murray,NaN,Dow Jones & Co.,"President, Dow Jones Leadership Inst.",connection,0.0,None,None
1,11936408,Aakriti,Beri,"Palo Alto, United States",Global Affairs Canada,Trade Commissioner,connection,0.0,None,None
2,12087185,Aaron,Adams,"Lansing, United States",CU Solutions Group,EVP/COO,connection,0.0,None,None
3,11871313,Aaron,Adolphson,"Santa Cruz , United States",Yodlee,"Principal Director, D&A Marketing",connection,0.0,None,None
4,11600052,Aaron,Axworthy,"Vancouver, Canada",Zafin,VP Strategic Sales,connection,0.0,None,None
5,11598844,Aaron,Bollinger,"ANDOVER,, United States",Voltage,CRO,connection,0.0,None,None
6,11803050,Aaron,Boyd,"New York, United States",Mastercard,Global Vice President,connection,0.0,None,None
7,11601236,Aaron,Byrne,"San Francisco, United States",L.E.K. Consulting,Head of Financial Services and FinTech,private,NaN,None,None
8,11598772,Aaron,Chesley,"Boise, United States",IXOPAY,"Director, Solutions Architecture",connection,0.0,None,None
9,11871401,Aaron,Coppock,"Los Angeles, United States",DAVE,Director of Software Engineering,connection,0.0,None,None


In [24]:
df_new.to_csv("money20-20_Attandees_Contacts.csv", index=False)

In [35]:
conditions = [
    df_new["email"].notna(),
    (df_new["email"].isna()) & (df_new["contact_sharing"] == "connection"),
    (df_new["email"].isna()) & (df_new["contact_sharing"] != "connection"),
]
choices = [
    "email available as connected",
    "email available if connected",
    "email not available as private",
]
df_new["email_availability"] = np.select(conditions, choices)
email_availability_counts = (
    df_new["email_availability"].value_counts(normalize=True) * 100
)
for category, percentage in email_availability_counts.items():
    print(f"For {percentage:.1f}%, {category}.")

For 60.0%, email available if connected.
For 40.0%, email not available as private.
For 0.1%, email available as connected.
